# 🧬 Práctica B: Anotación de Genomas con conda en Google Colab

> **Antes de continuar**, lea la [guía de prácticas compartida](00_genome_annotation_common.md) — contiene el contexto biológico de cada caso, los conceptos clave, las plataformas y la descripción de los archivos de salida.

Esta práctica sigue el mismo flujo de trabajo que la **Práctica A** (Galaxy), pero ejecutando las herramientas desde la línea de comandos en un entorno Google Colab + conda. Esto le permite:

- Instalar y usar herramientas bioinformáticas de CLI sin necesitar un servidor institucional.
- Integrar los resultados directamente en Python con `pandas` y `matplotlib`.
- Comprender cómo se encadenan estas herramientas en un pipeline automatizado.

---

## ⚙️ Paso 0 — Configurar el entorno

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 8–12 minutos.

In [1]:
# Instalamos biopython y Conda para colab
!pip install condacolab biopython --quiet

import condacolab
condacolab.install() # Esto reinicia el kernel automáticamente

# Verify condacolab installation
import os
os.environ['PATH'] = '/usr/local/bin:/opt/conda/bin:' + os.environ['PATH']

!conda --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.0 MB/s eta 0:00:00
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...
conda 24.11.2


Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para el ensamblaje.

In [2]:
# Instalar herramientas bioinformáticas

!conda install -y \
  -c bioconda \
  -c conda-forge \
  bakta \
  ncbi-amrfinderplus \
  plasmidfinder \
  integron_finder \
  isescan \
  antismash

!echo "✅ Herramientas bioinformáticas instaladas"

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: \ | / - \ done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - antismash
    - bakta
    - integron_finder
    - isescan
    - ncbi-amrfinderplus
    - plasmidfinder


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    about-time-4.2.1           |     pyhd8ed1ab_1          16 KB  conda-forge
    alive-progress-3.0.1       |     pyhd8ed1ab_0          62 KB  conda-forge
    antismash-8.0.4            |     pyhdfd78af_1        26.7 MB  bioconda
    aragorn-1.2.41             |       h7b50bb2_5         144 KB  bioconda
    attrs-26.1.0               |     pyhcf101f3_0          63 KB  conda-forge
    bakta-1.12.0               |     pyhdfd78af_0         104 KB  bioconda
    bcbio-gff-0.7.1            |     pyhdfd78af_3          24 KB  bioconda
    biopython-1.81         

ahora vamos a instalar otras herramientas propias de **Python** que nos ayudaran a procesar y ver los resutlados mas facil para los humanos

In [ ]:
!pip install matplotlib pandas --quiet

# Imports centralizados — todos en un solo lugar para evitar duplicación
import json
import os
import pathlib
import glob
import subprocess
from IPython.display import Image, display, SVG

import matplotlib.pyplot as plt
import pandas as pd

os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

print("✅ Dependencias Python listas")

## 📥 Paso 1 — Descargar los datos del caso asignado

Solo trabajaremos con un solo caso, por lo cual debe definir primero el caso que le indicó el profesor (A, B, C o D). Asigne a la variable el caso que va a trabajar.

Consulte la [guía compartida](00_genome_annotation_common.md#-casos-de-estudio) para el contexto biológico de cada caso.

In [3]:
# ⚠️ **IMPORTANTE**: Define el caso a trabajar aquí y solo aquí.
# Descomenta la línea que corresponda a tu caso (A, B, C o D).
CASO = "caso_A" # Por ejemplo, para caso A

In [4]:
# Crear directorio de datos
os.makedirs(f"{CASO}/data", exist_ok=True)

# Diccionario de URLs por caso
CASOS_CONFIG = {
    "caso_A": {
        "descripcion": "Staphylococcus aureus MRSA (~2.8 Mb, GC ~33%)",
        "url": "https://zenodo.org/records/17252812/files/DRR187559_contigs.fasta",
        "gz": False
    },
    "caso_B": {
        "descripcion": "Klebsiella pneumoniae (~5.5 Mb, GC ~57%)",
        "url": "https://zenodo.org/records/17252812/files/ERR14828471_contigs.fasta",
        "gz": False
    },
    "caso_C": {
        "descripcion": "Streptomyces venezuelae (~8.2 Mb, GC ~72%)",
        "url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/253/235/GCF_000253235.1_ASM25323v1/GCF_000253235.1_ASM25323v1_genomic.fna.gz",
        "gz": True
    },
    "caso_D": {
        "descripcion": "Pseudomonas abieticivorans (~6.7 Mb, GC ~63%)",
        "url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/023/509/015/GCF_023509015.1_ASM2350901v1/GCF_023509015.1_ASM2350901v1_genomic.fna.gz",
        "gz": True
    }
}

if CASO not in CASOS_CONFIG:
    raise ValueError(f"⚠️ CASO '{CASO}' no reconocido. Use 'caso_A', 'caso_B', 'caso_C' o 'caso_D'.")

cfg = CASOS_CONFIG[CASO]
print(f"── {CASO.upper()}: {cfg['descripcion']} ──")

contigs_path = f"{CASO}/data/contigs.fasta"

if cfg["gz"]:
    gz_path = f"{CASO}/data/contigs.fna.gz"
    print("Descargando contigs comprimidos...")
    !wget -q "{cfg['url']}" -O "{gz_path}"
    # FIX: verificar tamaño antes de descomprimir
    size = os.path.getsize(gz_path) if os.path.exists(gz_path) else 0
    print(f"  Descargado: {size:,} bytes")
    if size < 1000:
        raise RuntimeError(f"❌ Descarga fallida o archivo vacío: {gz_path}")
    !gunzip -f "{gz_path}"
    fna_path = f"{CASO}/data/contigs.fna"
    if os.path.exists(fna_path):
        os.rename(fna_path, contigs_path)
        print(f"Renombrado → {contigs_path}")
else:
    print("Descargando contigs...")
    !wget -q "{cfg['url']}" -O "{contigs_path}"
    size = os.path.getsize(contigs_path) if os.path.exists(contigs_path) else 0
    print(f"  Descargado: {size:,} bytes")
    if size < 1000:
        raise RuntimeError(f"❌ Descarga fallida o archivo vacío: {contigs_path}")

# Contar contigs
import subprocess
result = subprocess.run(["grep", "-c", ">", contigs_path], capture_output=True, text=True)
num_contigs = result.stdout.strip()
print(f"✅ Caso {CASO} listo — {num_contigs} contigs en {contigs_path}")


── CASO_C: Streptomyces venezuelae (~8.2 Mb, GC ~72%) ──
Descargando contigs comprimidos...
  Descargado: 2,287,319 bytes
Renombrado → caso_C/data/contigs.fasta
✅ Caso caso_C listo — 1 contigs en caso_C/data/contigs.fasta


## 👀 Paso 1.1 — Inspeccionar los archivos descargados

Vamos a verificar que los archivos FASTQ y el genoma de referencia FASTA se hayan descargado correctamente y tengan el formato esperado. Revisaremos el encabezado de cada uno.

In [5]:
print(f"=== Encabezado del archivo FASTA de contigs para {CASO} ===")
!head -n 10 "{contigs_path}"

print(f"\n=== Estadísticas básicas ===")
!grep -c '>' "{contigs_path}" | awk '{print "Contigs totales: " $1}'
!awk '/^>/{next}{total+=length($0)} END{print "Bases totales: " total}' "{contigs_path}"

=== Encabezado del archivo FASTA de contigs para caso_C ===
>NC_018750.1 Streptomyces venezuelae ATCC 10712, complete sequence
CCGGTCGACGAGCGGGCTTGTCCCCTGCCGGGCTGGTGCTTCTGGTGGTCGGGCCCGGCCGGGTGGTCAGGCTGAGAAGC
CGATGCTGGACACGTACTCGTACTGACCCCACTGGGAGCCGAGGTCCTGGACGACCTCACCAACCCAGATGTCGTCCATG
CCGGCATGGTCATCAGCAGCCCACGCGCCGATGATCCGGTCCCACCGACGAACGTTCAGGTGCCGGTACTCCACAACCCG
TTGGAGAGGACGTGCGACCTGGGACTGATTGAGGGGGTGGAACTCCACGCGTGTGCCGCGTCCCTGCCGGTTGAGACGGT
TCAGAAGGTACCGGGCGACGTTCTGACGCCGTACGGCACGATACGCAGACTCGATACGTTCCAAGTTCCGATGTGAAGGA
GCGCGCTTGCCCTCCCGCCAAGCTCTGATGGTCCGGTCCGTGACGGTCAGGCCAGCCGCACGCGCGGCCTGACGAGCGTG
CTCACTCCGGGTGAGATAGTTCAGCCTCGCCATCAGACCACGCCGCTGCTCAACCGGCGTCACCACAAAGCCCGCAAGCT
CATCCAGCTGACGCGCCACCAACTCATGACCCTTCACACCACGAGCCCCGAACTTCCCGAAATCAATCACCCGCTCCGGC
ACAACCCCACCCCCCATCCACCACTAAACCGCCCCCACAGCGAGAGCATGCCGTAAATACCTACACCCCCAAAATACCAG

=== Estadísticas básicas ===
grep: {contigs_path}: No such file or directory
awk: cannot open {contigs_path} (No such file or directory)


## 🏷️ Paso 2 — Anotación principal con Bakta

Bakta realiza la anotación estructural (CDS, ARNt, ARNr, ncRNA, CRISPR) y funcional en un solo paso.

> **Nota:** Bakta requiere su base de datos. La descargamos en formato light (~1 GB) — suficiente para esta práctica.

In [6]:
# Descargar base de datos de Bakta (versión light)

!mkdir -p /content/opt/bakta_db
!bakta_db download --output /content/opt/bakta_db --type light
print("✅ Base de datos Bakta lista")

Bakta software version: 1.12.0
Required database schema version: 6

Selected DB type: light

Fetch DB versions...
	... compatible DB versions: 1
Download database: v6.0, type=light, 2025-02-24, DOI: 10.5281/zenodo.14916843, URL: https://zenodo.org/record/14916843/files/db-light.tar.xz...
|████████████████████████████████████████| 1.34G/1.34G [100%] in 41.2s (32.58M/s
	... done
Check MD5 sum...
	...database file OK: 4a6e059ded39e9c5537ef4137d2f5648
Extract DB tarball: file=/content/opt/bakta_db/db-light.tar.xz, output=/content/opt/bakta_db
Successfully downloaded Bakta database!
	version: 6.0
	Type: light
	DOI: 10.5281/zenodo.14916843
	path: /content/opt/bakta_db/db-light
Update AMRFinderPlus database...
	... done

Run Bakta using '--db /content/opt/bakta_db/db-light' or set a BAKTA_DB environment variable: 'export BAKTA_DB=/content/opt/bakta_db/db-light'
✅ Base de datos Bakta lista


In [7]:
# Clear MPLBACKEND environment variable to avoid matplotlib backend conflicts
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

# Realizar la anotación con bakta
bakta_out = f"{CASO}/results/bakta_results"
os.makedirs(bakta_out, exist_ok=True)

!bakta \
  --db /content/opt/bakta_db/db-light \
  --output "{bakta_out}" \
  --force \
  --prefix genome \
  --keep-contig-headers \
  "{contigs_path}"

print(f"✅ Bakta completo. Resultados en {bakta_out}/")

Parse genome sequences...
	imported: 1
	filtered & revised: 1
	chromosomes: 1

Start annotation...
predict tRNAs...
	found: 74
predict tmRNAs...
	found: 1
predict rRNAs...
	found: 15
predict ncRNAs...
	found: 21
predict ncRNA regions...
	found: 26
predict CRISPR arrays...
	found: 0
predict & annotate CDSs...
	predicted: 7359 
	discarded length: 0
	discarded spurious: 0
	revised translational exceptions: 0
	skip UPS/IPS detection with light db version
	found PSCCs: 6390
	lookup annotations...
	conduct expert systems...
		amrfinder: 8
		protein sequences: 8
	combine annotations and mark hypotheticals...
	skip pseudogene detection with light db version
	analyze hypothetical proteins: 1132
		detected Pfam hits: 355 
		calculated proteins statistics
	revise special cases...
detect & annotate sORF...
	detected: 21972
	discarded due to overlaps: 17173
	discarded spurious: 0
	detected IPSs: 0
	found PSCCs: 0
	lookup annotations...
	filter and combine annotations...
	filtered sORFs: 0
detect ga

## 📊 Paso 3 — Explorar el resumen de Bakta con Python

In [8]:
txt_path = pathlib.Path(f"{bakta_out}/genome.txt")
print(txt_path.read_text())

Sequence(s):
Length: 8226158
Count: 1
GC: 72.5
N50: 8226158
N90: 8226158
N ratio: 0.1
coding density: 90.0

Annotation:
tRNAs: 73
tmRNAs: 1
rRNAs: 15
ncRNAs: 21
ncRNA regions: 26
CRISPR arrays: 0
CDSs: 7354
pseudogenes: 0
hypotheticals: 1130
sORFs: 0
gaps: 192
oriCs: 1
oriVs: 0
oriTs: 0

Bakta:
Software: v1.12.0
Database: v6.0, light
DOI: 10.1099/mgen.0.000685
URL: github.com/oschwengers/bakta



In [9]:
# Extraer métricas clave del resumen
txt = txt_path.read_text()
metricas = {}
for linea in txt.splitlines():
    if ":" in linea:
        clave, _, valor = linea.partition(":")
        metricas[clave.strip()] = valor.strip()

df_resumen = pd.DataFrame.from_dict(metricas, orient="index", columns=["Valor"])
print(df_resumen.to_string())

                                       Valor
Sequence(s)                                 
Length                               8226158
Count                                      1
GC                                      72.5
N50                                  8226158
N90                                  8226158
N ratio                                  0.1
coding density                          90.0
Annotation                                  
tRNAs                                     73
tmRNAs                                     1
rRNAs                                     15
ncRNAs                                    21
ncRNA regions                             26
CRISPR arrays                              0
CDSs                                    7354
pseudogenes                                0
hypotheticals                           1130
sORFs                                      0
gaps                                     192
oriCs                                      1
oriVs     

### Visualización del `circular plot` generado por Bakta

In [10]:
print(f"Archivos en {bakta_out}:")
!ls "{bakta_out}"

circular_plot_path = os.path.join(bakta_out, "genome.svg")

if os.path.exists(circular_plot_path):
    print(f"\nMostrando el circular plot: {circular_plot_path}")
    display(SVG(filename=circular_plot_path))
else:
    print("\nNo se encontró 'genome.svg'. Verifique que Bakta haya completado correctamente.")

Archivos en caso_C/results/bakta_results:
genome.embl  genome.fna   genome.hypotheticals.faa  genome.json  genome.svg
genome.faa   genome.gbff  genome.hypotheticals.tsv  genome.log	 genome.tsv
genome.ffn   genome.gff3  genome.inference.tsv	    genome.png	 genome.txt

Mostrando el circular plot: caso_C/results/bakta_results/genome.svg


**Preguntas:**

1. ¿Cuántos contigs había en el input?
2. ¿Cuál es la longitud total del borrador del genoma (*draft genome length*)?
3. ¿Cuántos CDS fueron predichos?
4. ¿Cuántas *small proteins* se encontraron?
5. ¿Cuántos ARNt y ARNr se anotaron?
6. ¿Hay secuencias CRISPR?
7. Compare sus resultados con la **Tabla 1 de Hikichi et al. 2019** (Caso A) o el artículo de referencia (Caso B). ¿Coinciden?

## 🦠 Paso 4 — Genes de resistencia con AMRFinderPlus

In [ ]:
# Descomente la línea correspondiente a su caso:
ORGANISMO = "Staphylococcus_aureus"   # Caso A
# ORGANISMO = "Klebsiella_pneumoniae"   # Caso B
# ORGANISMO = None  # Caso C — Streptomyces no tiene grupo taxonómico en AMRFinder
# ORGANISMO = "Pseudomonas_aeruginosa"  # Caso D

# Actualizar base de datos AMRFinder
!amrfinder -u
print("✅ Base de datos AMRFinder actualizada")

amr_out = f"{CASO}/results/amrfinder_results"
os.makedirs(amr_out, exist_ok=True)
amr_tsv = f"{amr_out}/amrfinder.tsv"

organism_flag = f"--organism {ORGANISMO}" if ORGANISMO else ""

!amrfinder \
  --nucleotide "{contigs_path}" \
  -o "{amr_tsv}" \
  {organism_flag} \
  --threads 2

print("✅ AMRFinderPlus completo")
print(f"\n--- Primeras 5 líneas de {amr_tsv} ---")
!head -n 5 "{amr_tsv}"

Running: amrfinder -u
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /usr/local/bin/
Software version: 4.2.7
Reverting to hard coded directory: /usr/local/share/amrfinderplus/data/latest
Running: /usr/local/bin/amrfinder_update -d /usr/local/share/amrfinderplus/data
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /usr/local/share/amrfinderplus/data/2026-03-24.1/
Running: /usr/local/bin/amrfinder_index /usr/local/share/amrfinderplus/data/2026-03-24.1/
Indexing
amrfinder_index took 6 seconds to complete
amrfinder_update took 12 seconds to complete
Database directory: /usr/local/share/amrfinderplus/data/2026-03-24.1
Database version: 2026-03-24.1
amrfinder took 12 seconds to complete
✅ Base de datos AMRFinder actualizada
Running: amrfinder --nucleotide caso_C/data/contigs.fasta -o caso_C/result

In [ ]:
amr_df = pd.read_csv(amr_tsv, sep="\t")
print(f"Total de genes encontrados: {len(amr_df)}\n")

# FIX: verificar que las columnas existen antes de seleccionarlas
cols_disponibles = [c for c in ["Element symbol", "Contig id", "Class", "Subclass", "% Identity to reference"] if c in amr_df.columns]
print(amr_df[cols_disponibles].to_string())

**Preguntas:**

8. ¿Qué genes de resistencia a antibióticos se encontraron?
9. ¿A qué familias de antibióticos pertenecen (columna `Class`)?
10. ¿En qué contigs se encuentran?
11. ¿Se encontraron factores de virulencia?

## 🔵 Paso 5 — Identificación de plásmidos con PlasmidFinder

In [ ]:
pf_out = f"{CASO}/results/plasmidfinder_results"
os.makedirs(pf_out, exist_ok=True)

# la ruta de la base de datos de PlasmidFinder varía según la versión instalada.
# Se detecta dinámicamente en lugar de usar una ruta fija hardcodeada.
pf_db_candidates = glob.glob("/usr/local/share/plasmidfinder*/database") + \
                   glob.glob("/opt/conda/share/plasmidfinder*/database")

if not pf_db_candidates:
    raise RuntimeError("❌ No se encontró la base de datos de PlasmidFinder. Verifique la instalación.")

pf_db = pf_db_candidates[0]
print(f"Usando base de datos: {pf_db}")

!plasmidfinder.py \
  -i "{contigs_path}" \
  -o "{pf_out}" \
  -p "{pf_db}" \
  -x

print("✅ PlasmidFinder completo")

In [ ]:
pf_tsv = f"{pf_out}/results_tab.tsv"
pf_df = pd.read_csv(pf_tsv, sep="\t")
print(f"Plásmidos encontrados: {len(pf_df)}\n")

# FIX: verificar columnas disponibles antes de seleccionarlas
cols_pf = [c for c in ["Plasmid", "Identity", "Query / Template length", "Contig", "Position in contig"] if c in pf_df.columns]
print(pf_df[cols_pf].to_string())

**Preguntas:**

12. ¿Cuántos plásmidos se identificaron?
13. ¿A qué replicones / familias pertenecen?
14. ¿En qué contigs se encuentran?
15. Cruzando con AMRFinderPlus: ¿algún gen de resistencia está en el mismo contig que un plásmido?

## 🔗 Paso 6 — Detección de integrones con IntegronFinder

In [ ]:
integron_out = f"{CASO}/results/integron_results"
os.makedirs(integron_out, exist_ok=True)

!integron_finder \
  --local-max \
  --func-annot "{contigs_path}" \
  --outdir "{integron_out}"

print("✅ IntegronFinder completo")

In [ ]:
summary_files = list(pathlib.Path(integron_out).glob("*.summary"))
if summary_files:
    print(summary_files[0].read_text())
else:
    print("⚠️ No se encontró archivo .summary. Verifique que IntegronFinder haya completado correctamente.")
    print(f"Archivos en {integron_out}:")
    !ls "{integron_out}"

**Preguntas:**

16. ¿Cuántos integrones completos se encontraron?
17. ¿Cuántos elementos In0 y CALIN?
18. ¿En qué contigs se localizan?

## 🔀 Paso 7 — Detección de elementos IS con ISEScan

In [ ]:
isescan_out = f"{CASO}/results/isescan_results"

!isescan.py \
  --seqfile "{contigs_path}" \
  --output "{isescan_out}" \
  --nthread 4

print("✅ ISEScan completo")

In [ ]:
is_files = glob.glob(f"{isescan_out}/*.tsv")
if is_files:
    is_df = pd.read_csv(is_files[0], sep="\t")
    print(f"Elementos IS detectados: {len(is_df)}\n")
    # FIX: verificar columnas disponibles (los nombres pueden variar entre versiones de ISEScan)
    cols_is = [c for c in ["family", "cluster", "seqID", "isBegin", "isEnd"] if c in is_df.columns]
    print(is_df[cols_is].to_string())
else:
    print("No se encontraron elementos IS o el archivo aún no está listo.")

**Preguntas:**

19. ¿Cuántos elementos IS se detectaron?
20. ¿En qué contigs se encuentran?
21. ¿Cuáles son las distintas familias de IS identificadas?

## 📊 Paso 8 — Resumen integrado con Python

Construye una tabla resumen que cruce los resultados de Bakta, AMRFinderPlus y PlasmidFinder por contig.

In [ ]:
# Cargar resultados
amr_df  = pd.read_csv(amr_tsv, sep="\t")
pf_df   = pd.read_csv(pf_tsv,  sep="\t")

# Contigs con genes de resistencia
amr_col = "Contig id" if "Contig id" in amr_df.columns else amr_df.columns[1]
pf_col  = "Contig"    if "Contig"    in pf_df.columns  else pf_df.columns[1]

amr_contigs     = set(amr_df[amr_col].dropna().unique())
plasmid_contigs = set(pf_df[pf_col].dropna().unique())
both            = amr_contigs & plasmid_contigs

print("=== Contigs con genes de resistencia ===")
print(sorted(amr_contigs) if amr_contigs else "Ninguno")
print("\n=== Contigs con plásmidos ===")
print(sorted(plasmid_contigs) if plasmid_contigs else "Ninguno")
print("\n=== Contigs con AMBOS (resistencia + plásmido) ===")
print(sorted(both) if both else "Ninguno")

## 🌿 Paso 9 — (Solo Casos C y D) Predicción de BGC con antiSMASH

> **Este paso es exclusivo para el Caso C (*Streptomyces venezuelae*) y D (*Pseudomonas abieticivorans*)**. Los Casos A y B pueden saltar directamente a las preguntas integradoras.

Los clústeres de genes biosintéticos (BGC) son regiones del genoma que codifican la maquinaria completa para sintetizar un metabolito secundario — antibióticos, antifúngicos, sideróforos, pigmentos, etc. **antiSMASH** es la herramienta estándar para su predicción. Su relevancia es diferente en cada caso:
- **Caso C** (*S. venezuelae*): género con la mayor diversidad de BGC conocida — antibióticos, antifúngicos, sideróforos.
- **Caso D** (*P. abieticivorans*): BGC para sideróforos (pioverdina), lipopéptidos y posibles rutas de biotransformación de diterpenos.

> ⚠️ antiSMASH no se instala con el bloque `conda install` anterior porque tiene dependencias que pueden entrar en conflicto. Se instala aquí de forma independiente.

In [ ]:
# Instalar antiSMASH por separado (puede tardar 5–10 minutos)
!conda install -y -c bioconda -c conda-forge antismash
print("✅ antiSMASH instalado")

In [ ]:
# Descargar base de datos de antiSMASH
!mkdir -p /content/opt/antismash_db
!download-antismash-databases --database-dir /content/opt/antismash_db
print("✅ Base de datos antiSMASH lista")

In [ ]:
antismash_out = f"{CASO}/results/antismash_results"
os.makedirs(antismash_out, exist_ok=True)

!antismash \
  "{contigs_path}" \
  --output-dir "{antismash_out}" \
  --taxon bacteria \
  --genefinding-tool prodigal \
  --pfam2go \
  --databases /content/opt/antismash_db \
  --cpus 4

print(f"✅ antiSMASH completo. Reporte en {antismash_out}/index.html")

### Explorar los resultados de antiSMASH con Python

In [ ]:
import re # Added import for regular expressions

json_files = list(pathlib.Path(antismash_out).glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No se encontró archivo JSON en {antismash_out}. ¿Completó antiSMASH?")

data = json.loads(json_files[0].read_text())

# Extraer información de todos los BGC encontrados
registros = []
for record in data.get("records", []):
    for feature in record.get("features", []):
        if feature.get("type") == "region":
            quals = feature.get("qualifiers", {})

            # Parse the location string
            location_str = feature["location"]
            match = re.match(r'\[(\d+):(\d+)\]\(.+\)', location_str)
            if match:
                start_coord = int(match.group(1))
                end_coord = int(match.group(2))
            else:
                start_coord = 0
                end_coord = 0 # Default values if parsing fails

            registros.append({
                "Contig":          record["id"],
                "BGC_type":        ", ".join(quals.get("product", ["desconocido"])),
                "Start":           start_coord,
                "End":             end_coord,
                "Longitud_pb":     end_coord - start_coord,
                "KnownCluster_hit": quals.get("knownclusterblast", ["—"])[0][:60]
                                    if quals.get("knownclusterblast") else "—"
            })

bgc_df = pd.DataFrame(registros)
print(f"Total de BGC predichos: {len(bgc_df)}\n")
print(bgc_df.to_string())

In [ ]:
# Distribución de tipos de BGC
tipo_counts = bgc_df["BGC_type"].value_counts()

fig, ax = plt.subplots(figsize=(8, 10))
tipo_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Número de BGC")
ax.set_title(f"Tipos de BGC predichos — {CASO}\n(antiSMASH, n={len(bgc_df)})")
ax.invert_yaxis()
plt.tight_layout()

bgc_plot = f"{antismash_out}/bgc_tipos_{CASO}.png"
plt.savefig(bgc_plot, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Gráfico guardado en {bgc_plot}")

**Preguntas — Caso C (antiSMASH):**

26. ¿Cuántos BGC predijo antiSMASH en el genoma de *S. venezuelae*?
27. ¿Qué tipos de BGC están presentes? ¿Cuál es el más frecuente?
28. ¿Cuál BGC corresponde al clúster del **cloranfenicol** según KnownClusterBlast?
29. ¿Hay BGC sin homología conocida en MIBiG? ¿Cuántos? ¿Qué implicaría caracterizar uno de ellos?
30. Elija un BGC que le llame la atención y describa: tipo, tamaño en pb y metabolito predicho.

**Preguntas — Caso D (*P. abieticivorans*):**

31. ¿Cuántos BGC predijo antiSMASH? ¿Cuántos tipos diferentes hay?
32. ¿Detectó antiSMASH algún clúster de **sideróforos** (ej. pioverdina)? ¿En qué posición del cromosoma?
33. Compare el número y diversidad de BGC entre los Casos C y D. ¿Cuál tiene mayor potencial biosintético y por qué?
34. ¿Qué relación podría haber entre la capacidad de *P. abieticivorans* de degradar diterpenos y la presencia de ciertos BGC?

In [ ]:
# Comprimir la carpeta de resultados de antiSMASH para descargarla
output_archive = f"{antismash_out}.zip"
!tar -czvf "{output_archive}" -C "{CASO}/results" antismash_results

print(f"\n✅ Carpeta de antiSMASH comprimida: {output_archive}")
print("Puedes descargarla desde el explorador de archivos de Colab.")

## 📦 Paso 10 — Comprimir todos los resultados

In [ ]:
all_results_archive = f"{CASO}/all_results.tar.gz"

# Comprimir toda la carpeta de resultados
!tar -czvf "{all_results_archive}" -C "{CASO}" results

print(f"\n✅ Todos los resultados comprimidos en: {all_results_archive}")
print("Puedes descargar este archivo desde el explorador de archivos de Colab.")

## ❓ Preguntas integradoras

**Casos A y B:**

35. Considerando los resultados de Bakta, AMRFinderPlus y PlasmidFinder: ¿qué puede concluir sobre la movilidad de los genes de resistencia?
36. Si un gen de resistencia está en un plásmido que también contiene un integrón, ¿qué implica para la diseminación de resistencias?
37. ¿Todos los contigs con plásmidos pertenecen claramente al organismo de estudio? ¿Cómo lo verificaría?
38. Para el **Caso B** (*K. pneumoniae*): ¿los resultados de AMRFinderPlus son consistentes con el perfil de resistencia a carbapenemes descrito en el artículo?

**Caso C y D:**

39. ¿Cuántos BGC identificó Bakta en su anotación estándar? ¿Coincide con los predichos por antiSMASH? ¿Por qué podrían diferir?
40. Compare el número de BGC predichos con lo reportado en la literatura para *S. venezuelae* ATCC 10712. ¿Cuántos tienen producto conocido y cuántos son "silenciosos" o sin homología en MIBiG?
41. ¿Qué ventaja tiene usar un genoma *finished* (como GCF_*.1) en lugar de un borrador (*draft*) con muchos contigs para este tipo de análisis?

---
*Consulte la [guía compartida](00_genome_annotation_common.md) y el [README del Módulo 6](../README.md) para repasar los conceptos.*
